# SpikeGate: Dynamic Neuromorphic Hardware Gating Analysis
This notebook runs the full `spikegate` analysis pipeline on the `MaxFormer` Spiking Neural Network natively in Google Colab.

In [1]:
!git clone https://github.com/SiddheshUttarwar/SNN-Transformer-Knowlege-Extraction.git
%cd SNN-Transformer-Knowlege-Extraction

# Install core dependencies
!pip install datasets torchvision spikingjelly timm

Cloning into 'SNN-Transformer-Knowlege-Extraction'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 152 (delta 35), reused 146 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 11.95 MiB | 38.00 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/SNN-Transformer-Knowlege-Extraction
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 14.9 MB/s eta 0:00:00


## 1. Load Model Checkpoint
Ensure your `10-384-T4.pth.tar` checkpoint is uploaded to Google Drive, then mount your drive and copy it to the `checkpoints` directory.

In [2]:
# Mount Google Drive if your checkpoint is there
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoints directory and copy the model
!mkdir -p checkpoints
!cp "/content/drive/MyDrive/10-384-T4.pth.tar" ./checkpoints/10-384-T4.pth.tar

# If your file path is different, adjust the cp command above!

from qkv_sparsity_extractor import load_maxformer
model, _, device = load_maxformer()
print(f"Model loaded on {device}")

Mounted at /content/drive
[1/4] Loading MaxFormer CPKT: ./checkpoints/10-384-T4.pth.tar
      Model loaded on: CUDA
Model loaded on cuda


## 2. Load Calibration Data
We use HuggingFace `datasets` to stream the ImageNet-1K validation set to avoid downloading the massive 150GB dataset into Colab.

In [3]:
import torch
from datasets import load_dataset
from torchvision import transforms

IMG_SIZE = 224
BATCH_SIZE = 10

dataset = load_dataset('mrm8488/ImageNet1K-val', split='train', streaming=True, trust_remote_code=True)
tfs = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def data_generator():
    b = []
    for item in dataset:
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        b.append(tfs(img).unsqueeze(0))
        if len(b) == BATCH_SIZE:
            yield torch.cat(b, dim=0), torch.zeros(BATCH_SIZE)
            b = []

gen = data_generator()
calibration_data = [next(gen) for _ in range(5)] # 50 samples for quick profiling

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mrm8488/ImageNet1K-val' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mrm8488/ImageNet1K-val' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json: 0.00B [00:00, ?B/s]

## 3. Run SpikeGate AutoProfiler
Automatically discovers MaxFormer's Spatio-Temporal output shape `(T, B, C, N)`, dynamically ungroups the 6 heads, and extracts detailed timing, sparsity, and ghost/dead neuron percentages.

In [4]:
from spikegate.profiler import AutoProfiler

# MaxFormer has T=4 timesteps and 6 heads per block
profiler = AutoProfiler(model, T=4)
profile_json = profiler.calibrate(
    calibration_data,
    device=device,
    num_blocks=10,
    num_heads=6,
    out_file="colab_gating_profile.json"
)

import json
print("Sample Policy for Block 0, Head 0:")
print(json.dumps(profile_json["block_0"]["head_0"], indent=2))

Starting Auto-Calibration for Spiking Gating...


100%|██████████| 5/5 [00:04<00:00,  1.05it/s]

Calibration Complete! Written policy to colab_gating_profile.json
Sample Policy for Block 0, Head 0:
{
  "sparsity_q_per_timestep": [
    0.7281,
    0.5375,
    0.7094,
    0.525
  ],
  "sparsity_k_per_timestep": [
    0.9969,
    0.9781,
    0.9938,
    0.9812
  ],
  "sparsity_v_per_timestep": [
    0.9625,
    0.8094,
    0.9563,
    0.8031
  ],
  "average_sparsity_q": 0.625,
  "stbp_qk_temporal_gap_abs": 1.222,
  "dead_neurons_pct_q": 44.06,
  "ghost_neurons_pct_q": 25.0,
  "dead_neurons_pct_k": 96.88,
  "ghost_neurons_pct_k": 0.0,
  "dead_neurons_pct_v": 77.19,
  "ghost_neurons_pct_v": 3.12,
  "is_highly_correlated": false,
  "HARDWARE_GATING_POLICY": "LATE_WAKEUP_GATE"
}


## 4. Run Head Importance Estimation
This runs the evaluator directly on the MaxFormer without modifying the base logic, using `MASK-ONE-OUT` and `ISOLATE-ONE` ablation strategies to identify the correlation between gating policies and actual model capability.

In [5]:
from spikegate.gating import DynamicGateController
from spikegate.evaluator import GatingAblationStudy

gate_ctrl = DynamicGateController("colab_gating_profile.json")

# We pass the model directly. To test *Compute Savings*, you'd need the SpikingGatedAttention wrappers.
# But for Head Importance Evaluation, passing the raw model will work because the Evaluator uses
# the Gate Controller's overrides to virtually "gate" the outputs if they were wrapped.
evaluator = GatingAblationStudy(model, gate_ctrl)

print("Fetching 5 batches for evaluation...")
eval_data = [next(gen) for _ in range(5)]

# Fast test on 2 blocks/6 heads.
# Set num_blocks=10 to run the full massive study!
importance_report = evaluator.run_head_importance_study(
    eval_data,
    device=device,
    num_blocks=2,
    num_heads=6,
    out_file="colab_head_importance.json",
    checkpoint_file="colab_head_importance_checkpoint.json"
)

print("\nAggregated Insights (How much fidelity drops per policy group):")
print(json.dumps(importance_report.get("AGGREGATED_INSIGHTS", {}), indent=2))

Fetching 5 batches for evaluation...

--- Starting Head Importance Estimation Study ---
[1/3] Running Baseline (No Gating) to establish Ground Truth...


Baseline: 100%|██████████| 5/5 [00:01<00:00,  3.43it/s]



[2/3] Running MASK-ONE-OUT Study (Disabling heads one by one)...


Mask B1H5: 100%|██████████| 5/5 [00:01<00:00,  3.41it/s]



[3/3] Running ISOLATE-ONE Study (Disabling all heads EXCEPT one)...


Isolate B1H5: 100%|██████████| 5/5 [00:01<00:00,  3.41it/s]


Head Importance Study Complete! Report saved to colab_head_importance.json

Aggregated Insights (How much fidelity drops per policy group):
{
  "MASK_ONE_OUT_AVG_IMPORTANCE_BY_POLICY": {
    "LATE_WAKEUP_GATE": 97.14,
    "STATICALLY_GATED_BY_REDUNDANCY": 75.0,
    "STATICALLY_PRUNE_OR_EARLY_EXIT_T1": 100.0
  },
  "ISOLATE_ONE_AVG_FIDELITY_BY_POLICY": {
    "LATE_WAKEUP_GATE": 17.14,
    "STATICALLY_GATED_BY_REDUNDANCY": 10.0,
    "STATICALLY_PRUNE_OR_EARLY_EXIT_T1": 40.0
  }
}
